# Wikidata Airport Attribute Profiling

#### Purposes
For each candidate Wikidata attribute, measure how many airports 
- (a) are missing a value, 
- (b) have exactly one value, or 
- (c) have 2+ conflicting values. 
This tells us whether an attribute is reliable enough to use directly (low conflict rate) or needs extra cleaning before we can rely on it for schema mapping / identity resolution against OurAirports and Kaggle.

#### Source:
Wikidata, queried via the [QLever](https://qlever.dev/wikidata) SPARQL endpoint (`https://qlever.dev/api/wikidata`), scoped to entities with `wdt:P31/wdt:P279* wd:Q1248784` (instance of: airport, **including subclasses** such as "international airport" / "commercial airport" / "domestic airport"). We initially used the strict `wdt:P31 wd:Q1248784` (no subclasses) and it silently excluded ~636 real airports typed only via a subclass — including Berlin Brandenburg Airport and Berlin Schönefeld Airport, so this isn't a fringe/negligible gap.

Source Paper URL: https://ad-publications.cs.uni-freiburg.de/CIKM_qlever_BB_2017.pdf

Everything needed to reproduce this analysis is self-contained in this notebook. The only external helper used is the generic HTTP wrapper `run_sparql_query`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.append(src_path)

from wdi_airports.ufreiburg_sparql import run_sparql_query

WIKIDATA_ENDPOINT = "https://qlever.dev/api/wikidata"

## Attribute Definitions

| Attribute | Wikidata property | Definition | 
|---|---|---|
| `lat` / `lon` | `wdt:P625` (via `geof:latitude`/`geof:longitude`) | Airport coordinates | |
| `elevation` | `wdt:P2044` | Elevation above sea level | |
| `located_in` | `wdt:P131` | Administrative territorial entity the airport is located in |
| `country` | `wdt:P17` | Country |
| `operator` | `wdt:P137` | Operating organization |
| `owner` | `wdt:P127` | Owning organization |
| `iata` | `wdt:P238` | IATA airport code | 
| `icao` | `wdt:P239` | ICAO airport code | 


## Methodology

For each attribute we run a **histogram query**: group by airport, count the number of *distinct* values that attribute has for that airport, then group again by that count. 

The output tells us, z.B., "902 airports have 0 values, 14,654 have exactly 1, 6 have 2+" — directly giving us missing / single-valued / conflict rates without needing to inspect individual rows.

**Why not just dump every value with `GROUP_CONCAT`?** We tried that first and it silently returns empty results on this QLever endpoint — confirmed with a minimal isolated test (same query, `COUNT(DISTINCT ?x)` correctly returns 4, `GROUP_CONCAT(DISTINCT ?x)` in the exact same row returns nothing at all, in both JSON and CSV output). This looks like a genuine bug/limitation in this QLever deployment, not a query-writing mistake — so we stick to `COUNT`-based histograms for profiling, and only fall back to raw, unaggregated `SELECT ?airport ?value` queries (no `GROUP BY`, no aggregate functions) when we need to inspect the actual conflicting values for a handful of specific airports (e.g. via a `VALUES` clause).

In [2]:
# Single-hop attributes: ?airport <property> ?x — fits the generic histogram template below.
SIMPLE_ATTRIBUTES = {
    "elevation": "wdt:P2044",
    "located_in": "wdt:P131",  
    "country": "wdt:P17",
    "operator": "wdt:P137",
    "owner": "wdt:P127",
    "iata": "wdt:P238",
    "icao": "wdt:P239",
    "faa": "wdt:P240",
}

build_query = lambda varName, propName: f"""
PREFIX wd:  <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>

SELECT ?{varName} (COUNT(?airport) AS ?num_airports)
WHERE {{
  SELECT ?airport (COUNT(DISTINCT ?x) AS ?{varName})
  WHERE {{
    ?airport wdt:P31/wdt:P279* wd:Q1248784 .
    OPTIONAL {{ ?airport {propName} ?x }}
  }}
  GROUP BY ?airport
}}
GROUP BY ?{varName}
ORDER BY ?{varName}
"""

# Coordinates are a special case: Wikidata stores them as one WKT Point (wdt:P625),
# so we need GeoSPARQL functions to split out latitude/longitude — doesn't fit the template above.
build_coord_query = lambda varName, axis: f"""
PREFIX wd:   <http://www.wikidata.org/entity/>
PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
PREFIX geof: <http://www.opengis.net/def/function/geosparql/>

SELECT ?{varName} (COUNT(?airport) AS ?num_airports)
WHERE {{
  SELECT ?airport (COUNT(DISTINCT ?x) AS ?{varName})
  WHERE {{
    ?airport wdt:P31/wdt:P279* wd:Q1248784 .
    OPTIONAL {{
      ?airport wdt:P625 ?coord .
      BIND(geof:{axis}(?coord) AS ?x)
    }}
  }}
  GROUP BY ?airport
}}
GROUP BY ?{varName}
ORDER BY ?{varName}
"""

# Timezone is a two-hop case: airports have no direct wdt:P421 (verified: 0% coverage).
# Instead we go airport --P131--> located-in place --P421--> timezone.
build_timezone_query = lambda varName: f"""
PREFIX wd:  <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>

SELECT ?{varName} (COUNT(?airport) AS ?num_airports)
WHERE {{
  SELECT ?airport (COUNT(DISTINCT ?x) AS ?{varName})
  WHERE {{
    ?airport wdt:P31/wdt:P279* wd:Q1248784 .
    OPTIONAL {{
      ?airport wdt:P131 ?loc .
      ?loc wdt:P421 ?x .
    }}
  }}
  GROUP BY ?airport
}}
GROUP BY ?{varName}
ORDER BY ?{varName}
"""

In [3]:
def _finalize(df: pd.DataFrame, var_name: str, attr_name: str, prop_name: str) -> pd.DataFrame:
    df = df.rename(columns={var_name: "n_distinct"})
    df["n_distinct"] = df["n_distinct"].astype(int)
    df["num_airports"] = df["num_airports"].astype(int)
    df.insert(0, "property", prop_name)
    df.insert(0, "attribute", attr_name)
    return df


def profile_attribute(attr_name: str, prop_name: str) -> pd.DataFrame:
    """Run the distinct-value-count histogram query for a single-hop Wikidata property."""
    var_name = f"n_distinct_{attr_name}"
    query = build_query(var_name, prop_name)
    df = run_sparql_query(query, endpoint=WIKIDATA_ENDPOINT)
    return _finalize(df, var_name, attr_name, prop_name)


def profile_coord_attribute(attr_name: str, axis: str) -> pd.DataFrame:
    """Run the distinct-value-count histogram query for lat/lon, extracted from wdt:P625."""
    var_name = f"n_distinct_{attr_name}"
    query = build_coord_query(var_name, axis)
    df = run_sparql_query(query, endpoint=WIKIDATA_ENDPOINT)
    return _finalize(df, var_name, attr_name, f"wdt:P625 (geof:{axis})")


def profile_timezone_attribute() -> pd.DataFrame:
    """Run the distinct-value-count histogram query for timezone, via the two-hop
    airport -> located_in (P131) -> timezone (P421) path. n_distinct here counts
    distinct timezone values reachable this way, so a conflict can also arise from
    located_in itself being multi-valued (e.g. city + country both carrying a
    timezone statement) — actual conflict resolution (picking one value) happens
    later in pandas, not here."""
    var_name = "n_distinct_timezone"
    query = build_timezone_query(var_name)
    df = run_sparql_query(query, endpoint=WIKIDATA_ENDPOINT)
    return _finalize(df, var_name, "timezone", "wdt:P131 -> wdt:P421")


def profile_attributes() -> pd.DataFrame:
    """
    Profile every attribute in SIMPLE_ATTRIBUTES plus lat/lon/timezone, stacked into one long DataFrame.
    """
    frames = []
    for attr_name, prop_name in SIMPLE_ATTRIBUTES.items():
        print(f"Profiling {attr_name} ({prop_name})...")
        frames.append(profile_attribute(attr_name, prop_name))
    for attr_name, axis in [("lat", "latitude"), ("lon", "longitude")]:
        print(f"Profiling {attr_name} (wdt:P625 via geof:{axis})...")
        frames.append(profile_coord_attribute(attr_name, axis))
    print("Profiling timezone (wdt:P131 -> wdt:P421)...")
    frames.append(profile_timezone_attribute())
    return pd.concat(frames, ignore_index=True)


def summarize_profile(profile_df: pd.DataFrame) -> pd.DataFrame:
    """
    Collapse the per-(attribute, n_distinct) histogram into one row per attribute with missing / single-valued / conflicting rates.
    """
    rows = []
    for (attr_name, prop_name), group in profile_df.groupby(["attribute", "property"]):
        total = group["num_airports"].sum()
        missing = group.loc[group["n_distinct"] == 0, "num_airports"].sum()
        single = group.loc[group["n_distinct"] == 1, "num_airports"].sum()
        conflict = group.loc[group["n_distinct"] >= 2, "num_airports"].sum()
        rows.append(
            {
                "attribute": attr_name,
                "property": prop_name,
                "total_airports": total,
                "missing": missing,
                "missing_rate": missing / total,
                "single_valued": single,
                "single_valued_rate": single / total,
                "conflict": conflict,
                "conflict_rate": conflict / total,
            }
        )
    summary = pd.DataFrame(rows).sort_values("conflict_rate", ascending=False).reset_index(drop=True)
    for col in ["missing_rate", "single_valued_rate", "conflict_rate"]:
        summary[col] = summary[col].map(lambda x: f"{x:.2%}")
    return summary

In [4]:
profile_df = profile_attributes()
summary_df = summarize_profile(profile_df)
summary_df

Profiling elevation (wdt:P2044)...
Profiling located_in (wdt:P131)...
Profiling country (wdt:P17)...
Profiling operator (wdt:P137)...
Profiling owner (wdt:P127)...
Profiling iata (wdt:P238)...
Profiling icao (wdt:P239)...
Profiling faa (wdt:P240)...
Profiling lat (wdt:P625 via geof:latitude)...
Profiling lon (wdt:P625 via geof:longitude)...
Profiling timezone (wdt:P131 -> wdt:P421)...


,attribute,property,total_airports,missing,missing_rate,single_valued,single_valued_rate,conflict,conflict_rate
0,timezone,wdt:P131 -> wdt:P421,25069,14855,59.26%,3217,12.83%,6997,27.91%
1,lat,wdt:P625 (geof:latitude),25069,112,0.45%,19788,78.93%,5169,20.62%
2,lon,wdt:P625 (geof:longitude),25069,112,0.45%,19954,79.60%,5003,19.96%
3,located_in,wdt:P131,25069,483,1.93%,23626,94.24%,960,3.83%
4,elevation,wdt:P2044,25069,6240,24.89%,18323,73.09%,506,2.02%
5,operator,wdt:P137,25069,22196,88.54%,2790,11.13%,83,0.33%
6,country,wdt:P17,25069,52,0.21%,24964,99.58%,53,0.21%
7,owner,wdt:P127,25069,24693,98.50%,348,1.39%,28,0.11%
8,icao,wdt:P239,25069,13490,53.81%,11568,46.14%,11,0.04%
9,iata,wdt:P238,25069,17091,68.18%,7976,31.82%,2,0.01%


## Findings

`country` and `located_in` both show strong coverage with low conflict rates (well under 4%), making them reliable enough to use directly for schema mapping / identity resolution. `operator`, `owner`, `iata`, and `icao` have much higher missing rates — expected, since not every airport has an operator/owner fact recorded, and for `iata`/`icao` specifically, many small airfields simply were never assigned a code in the real world (not a data quality gap to fix). `faa` (FAA airport code) is a solid extra matching key for US airports specifically: ~45% coverage, essentially zero conflicts.

`timezone` (via the two-hop `located_in -> timezone` path) reaches ~41% coverage, but with a much higher conflict rate (~28%) than the other attributes — likely because `located_in` itself is sometimes multi-valued (e.g. both a city and a country), and each of those carries its own timezone statement. We'll resolve this in pandas rather than in SPARQL (e.g. prefer the most specific `located_in` value's timezone) once we get to building the fused dataset.

**Open item**: `coordinates` conflict rate (~21.5%) is high for something that should be a single physical point. Needs a `VALUES`-based drill-down to check whether it's precision-only noise (e.g. two nearly-identical coordinate pairs) or a genuine data issue, before deciding how to pick a single lat/lon per airport for the final dataset.